# Memory, step 1: observations become a sequence
Goal: understand the recurrent backbone before adding the mixture-density prediction head. This is not yet a complete MDN-RNN or a training experiment.

Identical screenshots give identical latent distributions. A history can reveal motion that a current frame cannot. An LSTM learns a compressed history; it is not guaranteed to preserve every relevant fact.

In [ ]:
import torch
from torch import nn
torch.manual_seed(0)

## 1. Align observations and actions
For T actions, keep T+1 observations. Input at step t is `(z_t, a_t)` and its prediction target is `z_(t+1)`. All timesteps here belong to one episode per batch row. The random tensors are placeholders, not VAE encodings or real driving data.

In [ ]:
B, T = 2, 5
latent_dim, action_dim, hidden_dim = 32, 3, 256
z_sequence = torch.randn(B, T + 1, latent_dim)
actions = torch.rand(B, T, action_dim)
actions[..., 0] = 2 * actions[..., 0] - 1  # steering; gas and brake in [0,1]
current_z = z_sequence[:, :-1]
target_z = z_sequence[:, 1:]
inputs = torch.cat([current_z, actions], dim=-1)
print("Inputs:", inputs.shape, "Targets:", target_z.shape)
assert inputs.shape == (B, T, 35)
assert target_z.shape == (B, T, 32)

**Pause:** Why concatenate along the last dimension rather than the time dimension? Explain what each of the 35 numbers at one timestep represents.

## 2. Give the sequence to an LSTM
An LSTM carries two vectors: hidden state h (exposed output) and cell state c (internal memory). Learned gates control what is retained, written, and exposed. We will unpack the gates next.

Here h_t means the summary **before** consuming `(z_t,a_t)`; the LSTM output at index t is h_(t+1). A future prediction head will map that output to a distribution over z_(t+1). A controller at time t uses z_t and the pre-update h_t, avoiding any dependence on an action it has not chosen yet.

In [ ]:
memory = nn.LSTM(input_size=35, hidden_size=hidden_dim, batch_first=True)
h0 = torch.zeros(1, B, hidden_dim)
c0 = torch.zeros(1, B, hidden_dim)
outputs, (h_final, c_final) = memory(inputs, (h0, c0))
print("Output at every step:", outputs.shape)
print("Final hidden/cell:", h_final.shape, c_final.shape)
assert outputs.shape == (B, T, hidden_dim)
assert h_final.shape == c_final.shape == (1, B, hidden_dim)

## 3. Unroll the same computation one step at a time
The weights are reused at every timestep. Carrying state connects the steps. Reset state at independent episode boundaries; when later training across sequence chunks, we will discuss detaching state.

In [ ]:
with torch.no_grad():
    state = (h0, c0)
    steps = []
    for t in range(T):
        out, state = memory(inputs[:, t:t+1], state)
        steps.append(out)
    sequential_outputs = torch.cat(steps, dim=1)
    print("Sequence equals carried-state steps:", torch.allclose(outputs, sequential_outputs, atol=1e-6))
    assert torch.allclose(outputs, sequential_outputs, atol=1e-6)

## Stop and explain
1. Why would resetting h and c before every timestep remove access to earlier inputs?
2. Does an untrained LSTM already know velocity? Why not?
3. Why are the 256 outputs features, rather than a predicted 32-dimensional latent?

Next lesson: LSTM gates, then a mixture-density head, its likelihood loss, and sampling. Only after understanding those will we move memory into a reusable module.